Archived notebook excerpt: manuscript 100-gene branch. Outputs were removed. These cells preserve the recorded logic, including diagnostics; use the portable scripts in this directory for fresh execution. This is not evidence that training completed. Original source cell indices: 0, 1, 2.

In [ ]:
import os
# 1. 设置目标路径
# Set DMSO_SQUIDIFF_WORKDIR to your local working directory before use.
os.chdir(os.environ['DMSO_SQUIDIFF_WORKDIR'])

In [ ]:
import scanpy as sc
import pandas as pd
import os
import numpy as np

# ================= 1. 设置路径 =================
# 原始数据目录 (从 R 导出的那个文件夹)
raw_data_dir = "Radiation-data" 

# 你的基因列表文件 (请确保文件名一致，且放在当前目录下)
gene_list_file = "key_gene.csv"

# ================= 2. 读取你的 100 个基因 =================
print(f"正在读取基因列表: {gene_list_file} ...")
# header=None 表示没有列名，默认第一列索引是 0
gene_df = pd.read_csv(gene_list_file, header=None)
# 转成列表
my_genes = gene_df[0].values.tolist()

print(f"✅ 成功读取了 {len(my_genes)} 个目标基因。")
print(f"前 5 个基因预览: {my_genes[:5]}")

# ================= 3. 读取原始表达量数据 =================
print("\n正在读取原始单细胞数据 (这可能需要几秒钟)...")
# 读取矩阵 (注意转置 .T)
adata = sc.read_mtx(os.path.join( "matrix.mtx")).T
genes = pd.read_csv(os.path.join("genes.tsv"), header=None, sep='\t')
barcodes = pd.read_csv(os.path.join("barcodes.tsv"), header=None, sep='\t')

adata.var_names = genes[0].values
adata.obs_names = barcodes[0].values
adata.var_names_make_unique() # 防止基因重名报错

# 加载元数据 (Group 分组信息)
metadata = pd.read_csv(os.path.join( "metadata.csv"), index_col=0)
# 确保 metadata 和 matrix 的细胞顺序一致
adata.obs = metadata.loc[adata.obs_names]

# ================= 4. 标准预处理 =================
print("正在进行归一化 (Normalize) 和 对数化 (Log1p)...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ================= 5. 核心步骤：提取这 100 个基因 =================
# 检查匹配情况
valid_genes = [g for g in my_genes if g in adata.var_names]
missing_genes = [g for g in my_genes if g not in adata.var_names]

if len(missing_genes) > 0:
    print(f"\n⚠️  警告：有 {len(missing_genes)} 个基因在数据中未找到！")
    print(f"丢失的基因: {missing_genes}")
    print("可能原因：拼写不同（如大小写），或者这些基因在测序中完全没测到。")
else:
    print("\n✅ 完美！所有 100 个基因都找到了。")

# 真正的截取操作
if len(valid_genes) == 0:
    raise ValueError("错误：没有一个基因匹配成功，请检查 CSV 文件里的基因名格式！")

adata_subset = adata[:, valid_genes].copy()
print(f"截取后数据维度: {adata_subset.shape} (细胞数 x 基因数)")

# ================= 6. 拆分并保存 =================
# 确保输出目录存在
os.makedirs("Radiation-data", exist_ok=True)

# A. 制作训练集：只包含 Ctrl 和 IR
# 逻辑：模型只需要学习“正常 vs 辐射”的最强差异
train_obs = adata_subset.obs['Group'].isin(['Ctrl', 'IR'])
adata_train = adata_subset[train_obs].copy()

save_path_train = "train_targeted.h5ad"
adata_train.write(save_path_train)
print(f"\n✅ 训练数据已保存: {save_path_train}")
print(f"   包含组别: {adata_train.obs['Group'].unique()}")
print(f"   用于：train_squidiff.py 训练")

# B. 制作预测集：包含 DMSO 和 DMSO-IR (还有 Ctrl/IR 也可以带着)
# 逻辑：这是我们要输入模型去算向量、做预测的数据
# 这里我们把所有数据都存下来方便后续分析，或者你只存 DMSO 也可以
adata_predict = adata_subset.copy() 

save_path_pred = "predict_targeted.h5ad"
adata_predict.write(save_path_pred)
print(f"✅ 预测数据已保存: {save_path_pred}")
print(f"   包含所有组别，用于后续加载做预测分析")

print("\n准备工作完成！下一步：修改 gene_size 参数并开始训练。")

In [ ]:
!python train_squidiff.py \
    --logger_path logs/mouse_targeted_genes \
    --data_path train_targeted.h5ad \
    --resume_checkpoint mouse_targeted_100 \
    --gene_size 100 \
    --output_dim 100 \
    --batch_size 128 \
    --save_interval 5000 \
    --lr 1e-4